# 🚀 05: Results Aggregation and Pareto Analysis

*Part of the KV-Cache Eviction Capstone Series*
*Estimated time: 5 minutes*

---

We made it. This final notebook aggregates the artifacts produced by the preceding notebooks. It strictly processes **only observed** CSV artifacts, marks hardware-required runs explicitly, and exports publication-ready tables and plots.

### Experiment Roadmap
```text
[Artifacts] → [Validation] → [Observed-only tables] → [Pareto analysis] → [Report]
```


In [ ]:
NOTEBOOK_ID = '05_results_pareto_and_reporting'
REQUESTED_PROFILE = 't4'

# Colab bootstrap: install pinned dependencies and unpack the shared core.
# Upload kvcore_bundle.zip supplied with this notebook suite if kvcore is not present.
from pathlib import Path
import sys, subprocess, zipfile

PINNED = [
    'transformers==4.56.2', 'accelerate==1.10.1', 'datasets==4.0.0',
    'huggingface_hub==0.34.4', 'bitsandbytes==0.47.0', 'safetensors==0.6.2',
    'sentencepiece==0.2.1', 'scipy==1.16.1', 'matplotlib==3.10.6',
    'seaborn==0.13.2', 'pandas==2.3.2',
]
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *PINNED])

if not Path('kvcore').exists():
    try:
        from google.colab import files
        print('Upload kvcore_bundle.zip from the delivered suite.')
        uploaded = files.upload()
        archive = next((Path(name) for name in uploaded if name.endswith('.zip')), None)
        if archive is None:
            raise FileNotFoundError('Please upload kvcore_bundle.zip.')
        with zipfile.ZipFile(archive) as zf:
            zf.extractall('.')
    except ImportError as error:
        raise RuntimeError('Run in Google Colab or place the kvcore directory beside this notebook.') from error

sys.path.insert(0, str(Path('.').resolve()))
from kvcore import *
from kvcore.config import BENCHMARKS, MODELS, POLICY_DEFAULTS, PROFILES, SUITE_VERSION
print({'suite_version': SUITE_VERSION, 'ruler_revision': BENCHMARKS['ruler']['revision'], 'longbench_revision': BENCHMARKS['longbench']['revision']})


In [ ]:
from pathlib import Path
import pandas as pd, numpy as np, matplotlib.pyplot as plt, seaborn as sns, json

# Upload a ZIP of a completed run_root or mount Google Drive; no fabricated fallback data exists.
ARTIFACT_ROOT = Path('artifacts')
if not ARTIFACT_ROOT.exists():
    print('Place completed run artifacts beneath ./artifacts before aggregating. This notebook will not invent results.')


## Step 1: Why Does This Matter?

Data aggregation must be transparent. By strictly separating the execution notebooks (01–04) from the reporting notebook (05), we ensure that our final charts are drawn exclusively from traceable, observed artifacts.

Let us discover and validate the artifacts uploaded to the `artifacts/` directory.


In [ ]:
def find_csvs(root): return sorted(root.rglob('*.csv')) if root.exists() else []
files = find_csvs(ARTIFACT_ROOT)
print('Found CSV artifacts:', [str(f) for f in files])
frames = []
validated_sources = []
for path in files:
    try:
        # Validate the on-disk artifact before parsing. assert_artifact accepts a
        # file path, not an in-memory DataFrame or list of records.
        assert_artifact(path)
        frame = pd.read_csv(path)
        if {'policy', 'budget'}.issubset(frame.columns):
            frame['artifact_source'] = str(path)
            frames.append(frame)
            validated_sources.append(str(path))
    except Exception as error:
        print('Unreadable or invalid artifact:', path, repr(error))
if not frames:
    print('No compatible observed policy artifacts found. Run notebooks 02–04 and copy their run roots into artifacts/.')
results = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()
if not results.empty:
    # Notebook 04 intentionally writes a resumable-core CSV and a final-copy CSV.
    # Collapse only byte-identical measurements, retaining all materially distinct
    # artifacts from other notebooks/hardware tiers.
    identity = [name for name in ('benchmark','task','row_id','partition','policy','budget') if name in results.columns]
    value_columns = [name for name in results.columns if name != 'artifact_source']
    if identity:
        results = results.sort_values('artifact_source').copy()
        duplicate_mask = results.duplicated(subset=value_columns, keep='first')
        if duplicate_mask.any():
            print(f'Deduplicated {int(duplicate_mask.sum())} byte-equivalent policy records from final-copy exports.')
            results = results.loc[~duplicate_mask].copy()
    results.to_csv('aggregated_results.csv', index=False)
    write_json('aggregated_artifact_provenance.json', {
        'validated_sources': validated_sources,
        'raw_rows': int(sum(len(frame) for frame in frames)),
        'aggregated_rows': int(len(results)),
    })
    display(results.head())


## Step 2: Quality, Memory, and Latency Views

We generate plots only for metrics that were successfully observed. Status rows (like OOM fallbacks) remain in the tables for auditing but do not corrupt the data points.


In [ ]:
if not results.empty and 'status' in results:
    observed = results[results['status'].eq('observed')].copy()
else:
    observed = pd.DataFrame()

if not observed.empty:
    if {'budget','substring_match','policy'}.issubset(observed.columns):
        plt.figure(figsize=(9,4)); sns.lineplot(data=observed, x='budget', y='substring_match', hue='policy', marker='o'); plt.title('Observed quality proxy versus cache budget'); plt.tight_layout(); plt.savefig('quality_vs_budget.png', dpi=180); plt.show()
    if {'physical_kv_bytes_post_evict','substring_match','policy'}.issubset(observed.columns):
        plt.figure(figsize=(7,5)); sns.scatterplot(data=observed, x='physical_kv_bytes_post_evict', y='substring_match', hue='policy'); plt.title('Observed quality versus physical KV payload'); plt.tight_layout(); plt.savefig('quality_vs_memory.png', dpi=180); plt.show()
else:
    print('No observed points: dashboards intentionally remain empty.')


## Step 3: Pareto-Frontier Analysis

A policy/budget point is "nondominated" (on the Pareto frontier) if no other configuration can achieve equal or better quality using less memory and less time. Let us calculate the frontier for our test results.


In [ ]:
def pareto_flags(frame):
    required = {'physical_kv_bytes_post_evict','elapsed_seconds','substring_match'}
    if frame.empty or not required.issubset(frame.columns):
        return pd.Series(dtype=bool)
    flags = []
    values = frame.reset_index(drop=True)
    for i, row in values.iterrows():
        other = values.drop(i)
        dominates = (
            (other['physical_kv_bytes_post_evict'] <= row['physical_kv_bytes_post_evict']) &
            (other['elapsed_seconds'] <= row['elapsed_seconds']) &
            (other['substring_match'] >= row['substring_match']) &
            ((other['physical_kv_bytes_post_evict'] < row['physical_kv_bytes_post_evict']) |
             (other['elapsed_seconds'] < row['elapsed_seconds']) |
             (other['substring_match'] > row['substring_match']))
        ).any()
        flags.append(not dominates)
    return pd.Series(flags, index=frame.index)

if not observed.empty:
    observed['pareto_nondominated'] = pareto_flags(observed)
    observed.to_csv('aggregated_results_with_pareto.csv', index=False)
    print(observed['pareto_nondominated'].value_counts(dropna=False))
else:
    print('Pareto analysis requires observed results, not planned configurations.')


## Step 4: Final Reproducibility Checklist

### What to Look For
Before using these tables in the final report, verify that the output CSVs contain complete provenance: model revisions, GPU metadata, seeds, budget, and measurement class. 

Congratulations! The end-to-end evaluation is complete.
